# Episode 6: Team of Agents: Smart Speaker Selection

Round-robin gives everyone a turn. But what if the LLM could figure out who should speak next?

**In this episode, you'll build:** The same debate, but now an LLM decides who speaks.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen import ConversableAgent, LLMConfig

## From fixed order to smart selection

In Episode 5, our agents took predictable turns. That's fine for structured debates, but real conversations don't work that way. Sometimes the critic needs to jump in twice. Sometimes the synthesizer should wait until there's actually a disagreement to resolve.

**AutoPattern** uses an LLM-based group manager that reads the conversation and picks the most relevant agent to speak next. As Chi Wang, the creator of AG2, puts it: "If people practice using multi-agents, they often reach a solution faster." Smart routing is a big part of why.

The key to making this work well is a field you haven't seen yet: the agent's `description`.

---

### `description` vs `system_message` — here's a mistake I see constantly

New AG2 users mix these up all the time, and it quietly breaks their speaker selection. These two fields serve completely different audiences:

| Field | Who reads it | Purpose |
|-------|-------------|--------|
| `description` | The **group manager** (speaker selector) | Tells the manager what this agent is good at — **routing guidance** |
| `system_message` | The **agent itself** | Tells the agent how to behave — **execution guidance** |

Think of it this way: `description` is your agent's resume that the manager reads to decide who to call on. `system_message` is the private instructions the agent follows once it's called.

If you put execution instructions in `description`, the manager won't know when to pick this agent. If you put routing hints in `system_message`, the agent will behave oddly.

---

## Step 1: Create agents with descriptions

Same agents as Episode 5, but now each one has a `description` that tells the group manager when to call on them. Let's see this in action:

In [ ]:
from dotenv import load_dotenv

load_dotenv()


llm_config = LLMConfig({"model": "gpt-4o-mini"})

proposer = ConversableAgent(
    name="proposer",
    system_message=(
        "You are a tech strategist who PROPOSES architectural decisions. "
        "Present clear arguments with concrete benefits. Keep responses to 2-3 paragraphs."
    ),
    description="Proposes new ideas and architectural approaches. Speak to this agent when a new proposal or alternative is needed.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

challenger = ConversableAgent(
    name="challenger",
    system_message=(
        "You are a devil's advocate who CHALLENGES proposals. "
        "Find weaknesses, risks, and overlooked trade-offs. Be constructive but thorough. "
        "Keep responses to 2-3 paragraphs."
    ),
    description="Challenges and critiques proposals. Speak to this agent when an idea needs stress-testing or risk analysis.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

refiner = ConversableAgent(
    name="refiner",
    system_message=(
        "You are a solution architect who REFINES ideas. "
        "Synthesize the proposer's ideas and the challenger's concerns into improved solutions. "
        "Suggest concrete compromises. Keep responses to 2-3 paragraphs."
    ),
    description="Synthesizes and refines discussions. Speak to this agent when there are competing viewpoints that need resolution.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

## Step 2: Set up AutoPattern

The setup is almost identical to RoundRobin — the only addition is `group_manager_args`, which gives the speaker-selection LLM its own configuration.

In [ ]:
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import AutoPattern

user = ConversableAgent(name="user", llm_config=llm_config, human_input_mode="NEVER")

pattern = AutoPattern(
    initial_agent=proposer,
    agents=[proposer, challenger, refiner],
    group_manager_args={"llm_config": llm_config},
)

## Step 3: Run the debate

Same question, same agents, same number of rounds. The only difference is *who* decides the speaking order.

In [ ]:
result, context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Should we use microservices or monolith for our new project?",
    max_rounds=6,
)

## Compare with Episode 5

Look at the speaker order — it's different from the fixed round-robin. The LLM chose who should speak based on what the conversation actually needed at each moment.

You might see the challenger speak twice in a row if the proposer made multiple claims worth questioning. Or the refiner might jump in early if there's already enough tension to resolve. The conversation flows more naturally because the routing adapts to the content.

The trade-off is an extra LLM call per turn for speaker selection, and less predictability in the conversation flow.

## Trade-offs to consider

AutoPattern gives you flexibility, but flexibility has a price.

**It shines when** the task is open-ended — creative collaboration, brainstorming, exploratory analysis with diverse specialists. Anytime the next step genuinely depends on what was just said, smart selection produces better conversations than fixed ordering.

**A fixed pattern is better when** you need deterministic workflows, compliance-critical ordering (auditors love predictability), or when you already know the optimal agent order. It also saves cost — no extra LLM call for speaker selection.

## Mini-challenge

What happens when the group manager can't tell your agents apart? Try changing all three descriptions to `description="A helpful assistant"` and run the debate again.

Watch how the quality of speaker selection degrades. This is why **good descriptions are critical** for AutoPattern — they're the routing signals the manager relies on.

## Additional Content

### Tuning group_manager_args

The group manager that selects speakers is itself an LLM agent, and you can tune it independently. A practical trick: use a cheaper model for the agents themselves and a more capable model for the manager.

```python
# Use a more capable model for better speaker selection
manager_llm = LLMConfig({"model": "gpt-4o"})

pattern = AutoPattern(
    initial_agent=proposer,
    agents=[proposer, challenger, refiner],
    group_manager_args={"llm_config": manager_llm},
)
```

The manager only generates short selection decisions, so the cost of using a bigger model there is minimal. Meanwhile, your agents — who produce the bulk of the tokens — stay on the cheaper model.

## What's Next

So far, the LLM picks who speaks next from a pool. But what if agents could explicitly *hand off* to each other — like a customer support agent transferring you to a specialist? That's a different kind of control.

In the next episode, you'll learn **explicit routing** with handoff conditions, giving you the most precise control over conversation flow.

**Bonus:** Try the auto-pattern group chat demo in the [AG2 Playground](https://playground.ag2.ai).